# Django inspectdb, Mapping Existing Tables

## Introduction to inspectdb
---

In this lesson, you'll learn how to **generate Django models from existing database tables**.

This is essential when integrating Django with legacy databases or databases managed by other systems.

1. [The Legacy Database Scenario](#The-Legacy-Database-Scenario),
    - [When You Have Existing Tables](#When-You-Have-Existing-Tables),
    - [The inspectdb Approach](#The-inspectdb-Approach),
2. [Setting Up a Legacy Database](#Setting-Up-a-Legacy-Database),
    - [SQL Init Script](#SQL-Init-Script),
    - [Docker Compose with Init Script](#Docker-Compose-with-Init-Script),
3. [Using inspectdb](#Using-inspectdb),
    - [Basic Usage](#Basic-Usage),
    - [Inspecting Specific Tables](#Inspecting-Specific-Tables),
4. [Editing Generated Models](#Editing-Generated-Models),
    - [What to Always Check](#What-to-Always-Check),
    - [db_table and db_column](#db_table-and-db_column),
5. [Relations and Foreign Keys](#Relations-and-Foreign-Keys),
    - [How inspectdb Recognizes Relations](#How-inspectdb-Recognizes-Relations),
    - [Fixing Relation Issues](#Fixing-Relation-Issues),
6. [🧠 EXERCISE 🧠, Generate Models from Legacy Database](#🧠-EXERCISE-🧠,-Generate-Models-from-Legacy-Database).

<br>

## The Legacy Database Scenario

---

### When You Have Existing Tables

---

Common scenarios where you need to work with existing databases:

<br>

| Scenario | Description |
|----------|-------------|
| **Legacy migration** | Old PHP/Java app being replaced with Django |
| **Shared database** | Multiple apps use the same database |
| **Data warehouse** | Reporting database managed by DBA team |
| **Third-party system** | Database from vendor software |
| **Microservices** | Service needs read access to another service's DB |

<br>

### The inspectdb Approach

---

Django provides `inspectdb` command that:

1. **Connects** to your database
2. **Reads** table schemas
3. **Generates** Django model code

<br>

```
┌─────────────────────┐      inspectdb      ┌─────────────────────┐
│   Existing Tables   │  ─────────────────► │   Django Models     │
│   (SQL Schema)      │                     │   (Python Code)     │
└─────────────────────┘                     └─────────────────────┘
```

<br>

**Important:** `inspectdb` generates a *starting point* - you'll always need to review and edit the generated code!

<br>

## Setting Up a Legacy Database

---

For this lesson, we'll create a "legacy" database with tables that weren't created by Django.

<br>

### SQL Init Script

---

Create a folder `scripts/init-db/` and add this SQL file:

```sql
-- scripts/init-db/01-legacy-tables.sql
-- Legacy blog system
-- Tables use different naming convention than Django

CREATE TABLE IF NOT EXISTS articles (
    article_id INT IDENTITY(1,1) PRIMARY KEY,
    headline   NVARCHAR(200) NOT NULL,
    body       NVARCHAR(MAX),
    writer     NVARCHAR(100) NOT NULL,
    pub_date   DATE NOT NULL,
    created_at DATETIME2 DEFAULT GETDATE()
);

CREATE TABLE IF NOT EXISTS readers (
    reader_id  INT IDENTITY(1,1) PRIMARY KEY,
    username   NVARCHAR(100) NOT NULL UNIQUE,
    email      NVARCHAR(254) NOT NULL UNIQUE,
    joined_at  DATETIME2 DEFAULT GETDATE()
);

CREATE TABLE IF NOT EXISTS reader_reviews (
    review_id  INT IDENTITY(1,1) PRIMARY KEY,
    article_id INT NOT NULL REFERENCES articles(article_id),
    reader_id  INT NOT NULL REFERENCES readers(reader_id),
    score      SMALLINT NOT NULL,
    body       NVARCHAR(MAX),
    created_at DATETIME2 DEFAULT GETDATE(),
    CONSTRAINT unique_reader_review UNIQUE (article_id, reader_id)
);

-- Sample data
INSERT INTO articles (headline, writer, pub_date) VALUES
    ('Getting started with Django', 'Alice Brown', '2026-01-10'),
    ('REST APIs in Python',         'Bob Smith',  '2026-02-05'),
    ('Database design tips',        'Alice Brown', '2026-03-15');

INSERT INTO readers (username, email) VALUES
    ('jan_novak',  'jan@example.com'),
    ('petra_free', 'petra@example.com');

INSERT INTO reader_reviews (article_id, reader_id, score, body) VALUES
    (1, 1, 5, 'Excellent intro!'),
    (2, 1, 4, 'Very practical.'),
    (1, 2, 5, 'Loved it.');
```

<br>

### Docker Compose with Init Script

---

Update your `docker-compose.yml` to mount the init scripts:

```yaml
# docker-compose.yml

services:
  db:
    image: mcr.microsoft.com/mssql/server:2022-latest
    container_name: blog_db
    environment:
      ACCEPT_EULA: "Y"
      SA_PASSWORD: "Blog_pass1"
      MSSQL_PID: "Developer"
    ports:
      - "1433:1433"
    volumes:
      - mssql_data:/var/opt/mssql
      # Init scripts run on first startup
      - ./scripts/init-db:/docker-entrypoint-initdb.d

volumes:
  mssql_data:
```

<br>

**Start fresh (to run init scripts):**

```bash
# Remove existing data and restart
docker compose down -v
docker compose up -d

# Verify tables were created
docker compose exec db /opt/mssql-tools18/bin/sqlcmd \
  -S localhost -U SA -P "Blog_pass1" -No \
  -Q "SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_TYPE='BASE TABLE';"
```

<br>

You should see the legacy tables:
```
TABLE_NAME
--------------------
articles
readers
reader_reviews
```

<br>

## Using inspectdb

---

### Basic Usage

---

The basic syntax is:

```bash
python manage.py inspectdb
```

This outputs Django model code to stdout. To save it:

```bash
# Save to a new file
python manage.py inspectdb > catalog/models_legacy.py

# Or inspect and review first
python manage.py inspectdb | less
```

<br>

**Example output for our legacy tables:**

In [ ]:
# This is an auto-generated Django model module.
# You'll have to do the following manually to clean this up:
#   * Rearrange models' order
#   * Make sure each model has one field with primary_key=True
#   * Make sure each ForeignKey and OneToOneField has `on_delete` set
#   * Remove `managed = False` lines if you wish to allow Django to manage tables
# Feel free to rename the models, but don't rename db_table values or field names.

from django.db import models


class Articles(models.Model):
    article_id = models.AutoField(primary_key=True)
    headline = models.CharField(max_length=200)
    body = models.TextField(blank=True, null=True)
    writer = models.CharField(max_length=100)
    pub_date = models.DateField()
    created_at = models.DateTimeField(blank=True, null=True)

    class Meta:
        managed = False
        db_table = 'articles'


class Readers(models.Model):
    reader_id = models.AutoField(primary_key=True)
    username = models.CharField(unique=True, max_length=100)
    email = models.CharField(unique=True, max_length=254)
    joined_at = models.DateTimeField(blank=True, null=True)

    class Meta:
        managed = False
        db_table = 'readers'


class ReaderReviews(models.Model):
    review_id = models.AutoField(primary_key=True)
    article = models.ForeignKey(Articles, models.DO_NOTHING)
    reader = models.ForeignKey(Readers, models.DO_NOTHING)
    score = models.SmallIntegerField()
    body = models.TextField(blank=True, null=True)
    created_at = models.DateTimeField(blank=True, null=True)

    class Meta:
        managed = False
        db_table = 'reader_reviews'
        unique_together = (('article', 'reader'),)

<br>

### Inspecting Specific Tables

---

You can inspect only specific tables:

```bash
# Single table
python manage.py inspectdb employees

# Multiple tables
python manage.py inspectdb employees departments

# Exclude Django's own tables
python manage.py inspectdb | grep -v "django_"
```

<br>

## Editing Generated Models

---

### What to Always Check

---

The generated code is a **starting point**. Always review and fix:

<br>

| Issue | Problem | Fix |
|-------|---------|-----|
| **Model names** | Plural (`Employees`) | Rename to singular (`Employee`) |
| **Field names** | May be unclear | Add descriptive names |
| **on_delete** | Uses `DO_NOTHING` | Change to `CASCADE`, `PROTECT`, etc. |
| **blank/null** | Often too permissive | Review business rules |
| **Field types** | May be suboptimal | Use `EmailField`, etc. |
| **Meta options** | Missing `ordering`, `verbose_name` | Add as needed |

<br>

**Example: Cleaned up Employee model**

In [ ]:
# blog/models_legacy.py (after cleanup)

from django.db import models


class Article(models.Model):  # Singular name
    """Article from the legacy blog system."""
    article_id = models.AutoField(primary_key=True)
    headline = models.CharField(max_length=200, verbose_name='Headline')
    body = models.TextField(blank=True, null=True)
    writer = models.CharField(max_length=100, verbose_name='Writer')
    pub_date = models.DateField(verbose_name='Published date')
    created_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        managed = False
        db_table = 'articles'
        ordering = ['-pub_date']
        verbose_name = 'Article'
        verbose_name_plural = 'Articles'

    def __str__(self) -> str:
        return f"{self.headline} ({self.writer})"


class Reader(models.Model):  # Singular name
    """Reader from the legacy blog system."""
    reader_id = models.AutoField(primary_key=True)
    username = models.CharField(unique=True, max_length=100)
    email = models.EmailField(unique=True, max_length=254)  # EmailField!
    joined_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        managed = False
        db_table = 'readers'
        ordering = ['username']

    def __str__(self) -> str:
        return self.username

<br>

### db_table and db_column

---

These options map Django models to existing database names:

<br>

**`db_table`** - Maps model to table name:

```python
class Employee(models.Model):
    # ...
    
    class Meta:
        db_table = 'employees'  # Actual table name in DB
```

<br>

**`db_column`** - Maps field to column name:

```python
class Employee(models.Model):
    # Django field name is 'department', DB column is 'dept_id'
    department = models.ForeignKey(
        Department,
        db_column='dept_id',  # Actual column name in DB
        on_delete=models.PROTECT
    )
```

<br>

**When to use:**

| Situation | Solution |
|-----------|----------|
| Table name doesn't follow Django convention | Use `db_table` |
| Column name doesn't follow Django convention | Use `db_column` |
| Want nicer Python names | Rename field, add `db_column` |
| Primary key isn't `id` | Keep original field, add `primary_key=True` |

<br>

**Example: Using both db_table and db_column**

In [ ]:
class ReaderReview(models.Model):
    """Review left by a reader on an article."""
    review_id = models.AutoField(primary_key=True)

    # Rename FK fields to cleaner Python names, map to actual columns
    article = models.ForeignKey(
        Article,
        on_delete=models.CASCADE,
        db_column='article_id',   # Actual column name in DB
        related_name='reviews'
    )
    reader = models.ForeignKey(
        Reader,
        on_delete=models.CASCADE,
        db_column='reader_id',    # Actual column name in DB
        related_name='reviews'
    )

    # Rename 'score' to 'rating' in Python, map to actual column
    rating = models.SmallIntegerField(db_column='score')
    body = models.TextField(blank=True, null=True)
    created_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        managed = False
        db_table = 'reader_reviews'   # Maps to actual table
        unique_together = [['article', 'reader']]

    def __str__(self) -> str:
        return f"{self.reader} rated '{self.article.headline}' {self.rating}/5"

<br>

## Relations and Foreign Keys

---

### How inspectdb Recognizes Relations

---

`inspectdb` detects foreign keys from the database schema:

```sql
-- This SQL foreign key:
dept_id INTEGER REFERENCES departments(dept_id)

-- Becomes this Django field:
dept = models.ForeignKey(Departments, models.DO_NOTHING)
```

<br>

**What inspectdb does:**

| Database | Django |
|----------|--------|
| `FOREIGN KEY` constraint | `ForeignKey` field |
| `UNIQUE` constraint on FK | `OneToOneField` |
| Composite primary key | `unique_together` in Meta |
| `REFERENCES` | Sets related model |

<br>

### Fixing Relation Issues

---

**Issue 1: `DO_NOTHING` is dangerous**

```python
# BAD: Generated code
dept = models.ForeignKey(Departments, models.DO_NOTHING)

# GOOD: Choose appropriate behavior
department = models.ForeignKey(
    Department,
    on_delete=models.PROTECT,  # or CASCADE, SET_NULL
    db_column='dept_id'
)
```

<br>

| on_delete | Behavior |
|-----------|----------|
| `CASCADE` | Delete related objects |
| `PROTECT` | Prevent deletion if related objects exist |
| `SET_NULL` | Set FK to NULL (requires `null=True`) |
| `SET_DEFAULT` | Set to default value |
| `DO_NOTHING` | Do nothing (can cause integrity errors!) |

<br>

**Issue 2: Many-to-Many through table**

For junction tables (like `employee_projects`), you have two options:

In [ ]:
# Option 1: Keep as explicit model (if you need extra fields like 'body')

class ReaderReview(models.Model):
    """Junction table for reader-article reviews."""
    article = models.ForeignKey(
        Article,
        on_delete=models.CASCADE,
        db_column='article_id'
    )
    reader = models.ForeignKey(
        Reader,
        on_delete=models.CASCADE,
        db_column='reader_id'
    )
    rating = models.SmallIntegerField(db_column='score')
    body = models.TextField(blank=True, null=True)
    created_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        managed = False
        db_table = 'reader_reviews'
        unique_together = [['article', 'reader']]

In [ ]:
# Option 2: Add ManyToMany with 'through' (better for queries)

class Article(models.Model):
    # ... other fields ...

    # Add M2M field using the explicit through model
    readers = models.ManyToManyField(
        'Reader',
        through='ReaderReview',
        related_name='reviewed_articles'
    )

# Now you can do:
# article.readers.all()          # Get all readers who reviewed this article
# reader.reviewed_articles.all() # Get all articles reviewed by this reader

<br>

**Issue 3: Circular imports**

When models reference each other, use string references:

```python
# Instead of:
department = models.ForeignKey(Department, on_delete=models.PROTECT)

# Use string reference:
department = models.ForeignKey('Department', on_delete=models.PROTECT)
```

<br>

## Complete Cleaned Models Example

---

In [ ]:
# blog/models_legacy.py (complete cleaned version)

from django.db import models


class Article(models.Model):
    """Article from legacy blog system."""
    article_id = models.AutoField(primary_key=True)
    headline = models.CharField(max_length=200)
    body = models.TextField(blank=True, null=True)
    writer = models.CharField(max_length=100)
    pub_date = models.DateField()
    created_at = models.DateTimeField(auto_now_add=True)

    readers = models.ManyToManyField(
        'Reader',
        through='ReaderReview',
        related_name='reviewed_articles'
    )

    class Meta:
        managed = False
        db_table = 'articles'
        ordering = ['-pub_date']

    def __str__(self) -> str:
        return f"{self.headline} ({self.writer})"


class Reader(models.Model):
    """Reader from legacy blog system."""
    reader_id = models.AutoField(primary_key=True)
    username = models.CharField(unique=True, max_length=100)
    email = models.EmailField(unique=True, max_length=254)
    joined_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        managed = False
        db_table = 'readers'
        ordering = ['username']

    def __str__(self) -> str:
        return self.username


class ReaderReview(models.Model):
    """Review left by a reader on an article."""
    review_id = models.AutoField(primary_key=True)
    article = models.ForeignKey(
        Article,
        on_delete=models.CASCADE,
        db_column='article_id',
        related_name='reviews'
    )
    reader = models.ForeignKey(
        Reader,
        on_delete=models.CASCADE,
        db_column='reader_id',
        related_name='reviews'
    )
    rating = models.SmallIntegerField(db_column='score')
    body = models.TextField(blank=True, null=True)
    created_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        managed = False
        db_table = 'reader_reviews'
        ordering = ['-created_at']
        unique_together = [['article', 'reader']]

    def __str__(self) -> str:
        return f"{self.reader} rated '{self.article.headline}' {self.rating}/5"

<br>

## Testing the Models

---

After cleaning up the models, test them in Django shell:

```bash
python manage.py shell
```

```python
>>> from blog.models_legacy import Article, Reader, ReaderReview

>>> # List articles
>>> Article.objects.all()
<QuerySet [<Article: Database design tips (Alice Brown)>, ...]>

>>> # Get article with its reviews
>>> article = Article.objects.first()
>>> article.reviews.select_related('reader').all()
<QuerySet [<ReaderReview: jan_novak rated 'Database design tips' 5/5>]>

>>> # Filter by writer
>>> Article.objects.filter(writer='Alice Brown')
<QuerySet [<Article: Getting started with Django (Alice Brown)>, ...]>

>>> # Get all articles reviewed by a reader
>>> reader = Reader.objects.get(username='jan_novak')
>>> reader.reviewed_articles.all()
<QuerySet [<Article: Getting started with Django (Alice Brown)>, ...]>
```

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **inspectdb** | Generates Django models from existing database tables |
| **Basic usage** | `python manage.py inspectdb > models.py` |
| **Specific tables** | `python manage.py inspectdb table1 table2` |
| **db_table** | Maps model to table: `db_table = 'employees'` |
| **db_column** | Maps field to column: `db_column = 'dept_id'` |
| **Always fix** | Model names, `on_delete`, field types, `__str__` |
| **Relations** | inspectdb detects FKs but uses `DO_NOTHING` |
| **M2M tables** | Use explicit through model for extra fields |

<br>

### 🧠 EXERCISE 🧠, Generate Models from Legacy Database

---

Set up the legacy database and generate Django models:

1. Create `scripts/init-db/01-legacy-tables.sql` with the SQL script provided
2. Update `docker-compose.yml` to mount the init scripts
3. Restart database: `docker compose down -v && docker compose up -d`
4. Verify tables exist:
```bash
docker compose exec db /opt/mssql-tools18/bin/sqlcmd \
  -S localhost -U SA -P "Blog_pass1" -No \
  -Q "SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_TYPE='BASE TABLE';"
```
5. Run `python manage.py inspectdb > blog/models_legacy.py`
6. Edit the generated models:
   - Rename models to singular (`Articles` → `Article`)
   - Fix `on_delete` from `DO_NOTHING` to appropriate values
   - Add `__str__` methods
   - Add `db_column` where you rename fields (e.g. `score` → `rating`)
7. Test in Django shell

<details>
    <summary>▶️ Solution</summary>

**1-4. Setup (see instructions above)**

**5. Generate models:**

```bash
python manage.py inspectdb > blog/models_legacy.py
```

**6. Edit `blog/models_legacy.py`:**

```python
from django.db import models


class Article(models.Model):
    article_id = models.AutoField(primary_key=True)
    headline = models.CharField(max_length=200)
    body = models.TextField(blank=True, null=True)
    writer = models.CharField(max_length=100)
    pub_date = models.DateField()
    created_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        managed = False
        db_table = 'articles'

    def __str__(self):
        return f"{self.headline} ({self.writer})"


class Reader(models.Model):
    reader_id = models.AutoField(primary_key=True)
    username = models.CharField(unique=True, max_length=100)
    email = models.EmailField(unique=True, max_length=254)
    joined_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        managed = False
        db_table = 'readers'

    def __str__(self):
        return self.username


class ReaderReview(models.Model):
    review_id = models.AutoField(primary_key=True)
    article = models.ForeignKey(
        Article,
        on_delete=models.CASCADE,
        db_column='article_id'
    )
    reader = models.ForeignKey(
        Reader,
        on_delete=models.CASCADE,
        db_column='reader_id'
    )
    rating = models.SmallIntegerField(db_column='score')
    body = models.TextField(blank=True, null=True)
    created_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        managed = False
        db_table = 'reader_reviews'
        unique_together = [['article', 'reader']]

    def __str__(self):
        return f"{self.reader} rated '{self.article.headline}' {self.rating}/5"
```

**7. Test in shell:**

```python
>>> from blog.models_legacy import Article, Reader, ReaderReview
>>> Article.objects.all()
>>> Reader.objects.all()
>>> ReaderReview.objects.select_related('article', 'reader').all()
```

</details>

<br>

### 🧠 EXERCISE 🧠, Add M2M Relationship

---

Extend the models to support the many-to-many relationship:

1. Create the `Project` model from the `projects` table
2. Create the `EmployeeProject` model from the `employee_projects` table
3. Add `ManyToManyField` to `Employee` with `through='EmployeeProject'`
4. Test: Get all projects for an employee
5. Test: Get all employees on a project

<details>
    <summary>▶️ Solution</summary>
    
```python
# Add to catalog/models_legacy.py

class Project(models.Model):
    project_id = models.AutoField(primary_key=True)
    name = models.CharField(max_length=200, db_column='project_name')
    code = models.CharField(unique=True, max_length=20, db_column='project_code')
    start_date = models.DateField(null=True, blank=True)
    end_date = models.DateField(null=True, blank=True)
    budget = models.DecimalField(max_digits=12, decimal_places=2, null=True)
    status = models.CharField(max_length=20, default='planning')

    class Meta:
        managed = False
        db_table = 'projects'

    def __str__(self):
        return f"{self.code}: {self.name}"


class EmployeeProject(models.Model):
    employee = models.ForeignKey(
        Employee,
        on_delete=models.CASCADE,
        db_column='emp_id',
        primary_key=True
    )
    project = models.ForeignKey(
        Project,
        on_delete=models.CASCADE,
        db_column='project_id'
    )
    role = models.CharField(max_length=50, blank=True)
    assigned_date = models.DateField(auto_now_add=True)

    class Meta:
        managed = False
        db_table = 'employee_projects'
        unique_together = [['employee', 'project']]


# Update Employee model - add projects field:
class Employee(models.Model):
    # ... existing fields ...
    
    projects = models.ManyToManyField(
        Project,
        through='EmployeeProject',
        related_name='employees'
    )
```

**Test:**

```python
>>> from catalog.models_legacy import Employee, Project

>>> # Get employee's projects
>>> emp = Employee.objects.first()
>>> emp.projects.all()

>>> # Get project's employees
>>> proj = Project.objects.first()
>>> proj.employees.all()
```

</details>

---